# ERA V5 · Session 6 — Training Data Execution System
**Real data:** AI4Bharat **Sangraha `verified/tel`** (Telugu), top **10,000** rows.

**To run on Colab:** `File → Upload notebook →` this `run_pipeline.ipynb`, then drag
**`era_v5_tds.zip`** into the Files panel (left). Then **Runtime → Run all**. Cell 1
unzips the repo automatically.

The notebook (1) downloads the top‑10k rows once and **pins** them to
`data/sangraha_tel_10k.jsonl.gz`, then (2) runs the exact command `python run_demo.py`,
which reads that pinned cache. Pinned data + determinism ⇒ **you and the grader get
identical `submission_artifacts/`** (only wall‑clock throughput differs).


In [ ]:
# 1 · Setup — unzip the uploaded repo (if needed), install deps, set the path
import os, sys, glob, zipfile
if not os.path.exists("run_demo.py"):
    for z in glob.glob("*.zip") + glob.glob("/content/*.zip"):   # you dropped era_v5_tds.zip in Files
        try:
            with zipfile.ZipFile(z) as f: f.extractall(".")
        except Exception: pass
    for c in ("era_v5_tds", "/content/era_v5_tds"):
        if os.path.exists(os.path.join(c, "run_demo.py")):
            os.chdir(c); break
!pip -q install datasets
assert os.path.exists("run_demo.py"), "Upload era_v5_tds.zip to the Files panel, then re-run this cell."
sys.path.insert(0, os.getcwd()); print("repo root:", os.getcwd())

repo root: /content/era_v5_tds


In [ ]:
from huggingface_hub import login

login()

In [ ]:
# 2 · Download top-10k Sangraha verified/tel and PIN it to data/ (commit this file)
from tds import datasource as ds
N_ROWS = 10000
try:
    records = ds.load_sangraha_telugu(N_ROWS)      # streams from HuggingFace
    print(f"Loaded {len(records)} Telugu rows from ai4bharat/sangraha :: verified/tel")
except Exception as e:
    print("HF load failed (%r) — using bundled fixture instead." % e)
    records = ds.fixture_records()
path = ds.save_cache(records)                      # -> data/sangraha_tel_10k.jsonl.gz
print("pinned dataset ->", path, f"({os.path.getsize(path)//1024} KB)  <-- commit this")

README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loaded 10000 Telugu rows from ai4bharat/sangraha :: verified/tel
pinned dataset -> data/sangraha_tel_10k.jsonl.gz (11071 KB)  <-- commit this


In [ ]:
# 3 · Run the EXACT submitted command (reads the pinned cache -> real 10k artifacts)
!python run_demo.py


== ERA V5 · TRAINING DATA EXECUTION SYSTEM
[   0.000s] seed=20250804  (single global seed => full reproducibility)

== TOKENIZER  (frozen, hashed — gives token IDs meaning)
[   0.000s] indic-lane data: 10000 rows from pinned cache; shard pool ≤ 1500 docs; seq_len=64; steps=24
[PASS] tokenizer_hash_verified :: tok_f17abb75fa3d reproduces across builds (vocab=40000)
[PASS] tokenizer_hash_changes :: hash changes iff the vocabulary changes

== SHARDS + MANIFESTS  (immutable tokenized objects)
[   2.934s] shards created: 113 immutable tokenized shards, 445674 tokens, lanes=['agentic', 'code', 'eval_holdout', 'general_web', 'indic', 'math_science', 'reasoning']
[PASS] manifests_written :: 113 shards; 445674 tokens; lanes=['agentic', 'code', 'eval_holdout', 'general_web', 'indic', 'math_science', 'reasoning']
[   3.118s] manifests validated: 113/113 content + tokenizer hashes match
[PASS] manifests_validated :: every manifest's content_hash recomputes from its tokens; tokenizer_hash matches


In [ ]:
# 4 · View the generated pipeline dashboard inline + zip everything for download
from IPython.display import HTML, FileLink
import shutil
shutil.make_archive("era_v5_tds_submission", "zip", ".")   # code + data/ + submission_artifacts/
display(FileLink("era_v5_tds_submission.zip"))
display(FileLink("submission_artifacts/evidence.html"))
HTML(open("submission_artifacts/evidence.html", encoding="utf-8").read())

/content/era_v5_tds/era_v5_tds_submission.zip

/content/era_v5_tds/submission_artifacts/evidence.html

Requirement,Result,Evidence,Detail
Tokenizer integrity,PASS,manifests/tokenizer.json,Frozen hash reproduces; changes iff vocab changes
Shard immutability,PASS,manifests/,Editing a shard yields a new content hash + lineage; manifests revalidated
Evaluation firewall,PASS,run.log,never_train + canary shards blocked from loss-bearing batches
Packing correctness,PASS,run.log,No cross-doc attention; response-only loss; position reset
Mixture compliance,PASS,manifests/schedule.json + ledgers/consumption.jsonl,Planned mixture vs realized lane shares; scarce lanes held above floor
OPUS audit trail,PASS,ledgers/opus.jsonl,accept/reject/defer + protected-floor override recorded
Learning trace,FAIL,ledgers/learning.jsonl,Per-shard loss linked to source; already-learned flagged
Crash recovery,PASS,ledgers/consumption.jsonl,Next batch after resume is exact; no skip/repeat
Replay,PASS,ledgers/consumption.jsonl,Replayed interval reproduces identical hashes + spans
Fork,PASS,checkpoints/fork_lineage.json,Branch from earlier checkpoint diverges w/ lineage


In [ ]:
# 5 · (optional) automated invariant tests
!python -m pytest -q